# DoG submission — self-contained

No torch, no `tracksdata`, no network. `numpy` + `scipy` + `zarr` only.

**Before committing:** replace `T_TRUE_Q / T_TRUE_A / T_TRUE_B` in cell 1 with the
fitted values from `reports/t_true_model.json`. As shipped they are identity
placeholders and the node budget will be wrong.

Test volumes carry no `.geff`, so `estimated_number_of_nodes` is unavailable and
the node budget comes from a log-linear fit on the stored intensity quantiles.

In [ ]:
# zarr is NOT preinstalled in the Kaggle image (measured by probe kernel: torch
# 2.10, numpy 2.0.2, scipy 1.16.3, polars — no zarr). Internet is disabled, so it
# comes from the offline wheels dataset. numpy and scipy are already present.
import subprocess, sys
from pathlib import Path

wheels = next(p.parent for p in Path("/kaggle/input").rglob("zarr-*.whl"))
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                f"--find-links={wheels}", "zarr"], check=True)
import zarr
print("zarr", zarr.__version__)
assert int(zarr.__version__.split(".")[0]) >= 3, "need zarr 3.x for Zarr v3 stores"


In [ ]:
# === Cell 1: constants ===
import csv, os, time
from pathlib import Path

import numpy as np
import zarr
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment

DOWNSAMPLE = (1, 4, 4)     # strided; scale * downsample -> isotropic grid
ISO_UM     = 1.625         # um per voxel on the downsampled grid
DOG_RATIO  = 1.6           # approximates a Laplacian of Gaussian
BIG        = 1.0e9

# --- tuned on LOEO CV, gate re-swept at the saturated budget ---
GATE_UM    = 6.0
# Divisions deliberately OFF. Measured on the full 71-sample fold with this
# detector: no second pass 0.5707, second pass at 6.0 um 0.5693. It created
# 74,099 forks for 0.0040 of division Jaccard -- worth 0.0004 of score -- and
# gave up 0.0017 of edge Jaccard doing it. This config emits ~13x the nodes
# ours does (ratio +0.07 vs -0.33), so the second pass has far more spurious
# candidates to attach; the same gate is marginally POSITIVE on a sparser
# detector. Leaving it off is correct here, not an oversight.
DIV_GATE_UM = None           # 8.0 was fitted for a 26k-node detector
BUDGET     = 1.0           # set from the budget sweep
DENSE_CPF  = 100.0         # estimated cpf runs low; rule gain flat over 100-150
SIGMA_DENSE,  SEP_DENSE  = 2.0,  3.2
SIGMA_SPARSE, SEP_SPARSE = 2.93, 5.86

# Fitted over all 199 samples; reports/t_true_model.json does not exist,
# so these come from src/volt_biohub/detectors.py, which carries the same fit.
# The identity placeholders (A=0, B=1) gave 5 and 16 nodes/frame against
# 258 and 698 needed -- 2% of target, a score near zero.
T_TRUE_Q = "0.9"
T_TRUE_A = 4.57252748   # fitted: detectors.N_TRUE_LOG_INTERCEPT
T_TRUE_B = 0.79920777   # fitted: detectors.N_TRUE_LOG_SLOPE

In [ ]:
# === Cell 2: pipeline ===
def peak_window(sep_um, iso_um):
    """Odd maximum_filter size enforcing >= sep_um. Floors at 3."""
    need = max(1, int(np.ceil(sep_um / iso_um)))
    return max(3, (2 * need - 1) | 1)


def top_peaks(field, n_keep, window):
    """Local maxima, strongest first, truncated to n_keep."""
    if n_keep <= 0:
        return np.empty((0, 3), np.int64)
    is_peak = (field >= maximum_filter(field, size=window, mode="nearest")) & (field > 0)
    idx = np.argwhere(is_peak)
    if len(idx) == 0:
        return np.empty((0, 3), np.int64)
    vals = field[idx[:, 0], idx[:, 1], idx[:, 2]].astype(np.float64)
    if len(idx) > n_keep:
        # Deterministic jitter spreads ties across a plateau instead of taking
        # its first voxels in raster order.
        vals = vals + np.random.default_rng(0).uniform(0, 1e-9, len(vals))
    return idx[np.argsort(-vals)[:n_keep]].astype(np.int64)


def detect(vol, n_keep, sigma_um, sep_um):
    s = sigma_um / ISO_UM
    dog = gaussian_filter(vol, s) - gaussian_filter(vol, s * DOG_RATIO)
    return top_peaks(dog, n_keep, peak_window(sep_um, ISO_UM))


def link(a, b, gate_um, div_gate_um=DIV_GATE_UM):
    """Hungarian assignment on physical distance. Only t -> t+1 edges.

    The second pass is what makes the division term reachable at all. Hungarian
    assignment is one-to-one, so with no second pass the prediction contains no
    forks and division Jaccard is exactly 0 -- forfeiting the whole 0.1 weight.
    Our first submission did that: 0 forks in 73,130 edges. Re-swept on fold-0
    samples that actually contain divisions, 6.0 um is the best a distance-ranked
    ranker can do; opening it to 10 um finds twice the real forks and scores worse,
    drowned by 390 false ones.
    """
    if len(a) == 0 or len(b) == 0:
        return []
    d = np.linalg.norm((a[:, None, :].astype(np.float64)
                        - b[None, :, :].astype(np.float64)) * ISO_UM, axis=-1)
    cost = np.where(d <= gate_um, d, BIG)
    rows, cols = linear_sum_assignment(cost)
    out = [(int(i), int(j)) for i, j in zip(rows, cols) if cost[i, j] < BIG]
    if div_gate_um is None or not out:
        return out

    # The far daughter of a division loses the 1-to-1 assignment to the near one,
    # so it arrives here as an orphaned target. Closest pair first, <=2 children.
    kids = {}
    for i, _ in out:
        kids[i] = kids.get(i, 0) + 1
    claimed = {j for _, j in out}
    cand = sorted((d[i, j], i, j)
                  for j in range(len(b)) if j not in claimed
                  for i in kids if d[i, j] <= div_gate_um)
    for _dist, i, j in cand:
        if j in claimed or kids.get(i, 0) >= 2:
            continue
        out.append((i, j))
        claimed.add(j)
        kids[i] += 1
    return out


def dataset_meta(zp):
    """(estimated t_true, n_frames, lo, hi) from the image attrs alone."""
    g = zarr.open_group(str(zp), mode="r")
    q = g.attrs["image_statistics"]["quantiles"]
    t_est = float(np.exp(T_TRUE_A + T_TRUE_B * np.log(float(q[T_TRUE_Q]))))
    return t_est, int(g["0"].shape[0]), float(q["0.001"]), float(q["0.999"])


def load_frame(arr, t, lo, hi):
    """One normalised frame. Matches the training path: clamp low, no upper bound."""
    raw = np.asarray(
        arr[t, ::DOWNSAMPLE[0], ::DOWNSAMPLE[1], ::DOWNSAMPLE[2]]
    ).astype(np.float32)
    return np.clip((raw - lo) / (hi - lo + 1e-6), 0, None)

In [ ]:
# === Cell 3: run ===
roots = [Path("/kaggle/input") / d for d in os.listdir("/kaggle/input")]
test_dir = next(p for r in roots for p in r.rglob("*")
                if p.is_dir() and list(p.glob("*.zarr")) and "test" in p.name.lower())
zarrs = sorted(test_dir.glob("*.zarr"))
print(f"{len(zarrs)} datasets from {test_dir}", flush=True)

up = np.asarray(DOWNSAMPLE)
rid = nid = 0
t0 = time.time()

with open("submission.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
                "source_id", "target_id"])

    for k, zp in enumerate(zarrs, 1):
        name = zp.stem
        t_est, nt, lo, hi = dataset_meta(zp)
        cpf = t_est / max(1, nt)
        sigma, sep = ((SIGMA_DENSE, SEP_DENSE) if cpf >= DENSE_CPF
                      else (SIGMA_SPARSE, SEP_SPARSE))
        n_per_frame = max(1, round(BUDGET * t_est / max(nt, 1)))

        arr = zarr.open_group(str(zp), mode="r")["0"]
        prev_c = prev_ids = None
        n_nodes = n_edges = 0

        for t in range(nt):
            # Per-frame read keeps peak RAM near 10 MB instead of ~840 MB.
            c = detect(load_frame(arr, t, lo, hi), n_per_frame, sigma, sep)

            ids = []
            for zyx in np.rint(c * up).astype(int):
                nid += 1
                ids.append(nid)
                w.writerow([rid, name, "node", nid, t, *zyx.tolist(), -1, -1])
                rid += 1
            n_nodes += len(ids)

            if prev_c is not None:
                for i, j in link(prev_c, c, GATE_UM):
                    w.writerow([rid, name, "edge", -1, -1, -1, -1, -1,
                                prev_ids[i], ids[j]])
                    rid += 1
                    n_edges += 1
            prev_c, prev_ids = c, ids

        el = time.time() - t0
        print(f"[{k:4d}/{len(zarrs)}] {name} cpf={cpf:7.1f} sigma={sigma} "
              f"nodes={n_nodes:6d} edges={n_edges:6d} | {el/k:5.1f}s/ds "
              f"eta {el/k*(len(zarrs)-k)/3600:5.2f}h", flush=True)

print(f"\n{rid} rows, {(time.time()-t0)/60:.1f} min -> submission.csv")

In [ ]:
# === Cell 4: sanity check ===
import collections

with open("submission.csv") as fh:
    rows = list(csv.DictReader(fh))

ds = collections.Counter(r["dataset"] for r in rows)
kinds = collections.Counter(r["row_type"] for r in rows)
print("rows:", len(rows), "| datasets:", len(ds), "| expected:", len(zarrs))
print("row types:", dict(kinds))
print("ids consecutive:", [int(r["id"]) for r in rows[:3]] == [0, 1, 2]
      and int(rows[-1]["id"]) == len(rows) - 1)
print("all datasets present:", set(ds) == {p.stem for p in zarrs})
print("min nodes per dataset:", min(
    sum(1 for r in rows if r["dataset"] == d and r["row_type"] == "node") for d in ds))